## Executive Summary

This notebook implements the **star-of-slots (SOS) method** — the canonical winding layout engine for AC machines.

### What it does
Given a slot/pole/phase combination (Q, P, m), the SOS method:
1. Projects each slot's EMF phasor onto a circle divided into 2m sectors.
2. Assigns each slot conductor to a phase and current direction based on which sector its phasor falls in.
3. Builds the coil matrix used by winding factor and MMF analysis.

### When to use this module
- **Winding layout**: understand which slots carry which phase in which direction.
- **Feasibility check**: verify that a given (Q, P, m) combination supports a valid balanced winding.
- **Input to winding factors and MMF**: `build_coil_matrix` is the single output consumed by `winding_factors.py` and `mmf.py`.

> **Swappability:** this module is deliberately isolated as the layout engine. Future layout techniques (e.g. optimal assignment, genetic algorithm) can replace it without changing `winding_factors.py` or `mmf.py`.

## How It Fits Into Motor Design

```
01_winding_sos.ipynb   ←── YOU ARE HERE
(winding layout)
      │
      ▼
01_winding_factors.ipynb   (kp, kd, kw, harmonic spectrum)
      │
      ▼
01_winding_mmf.ipynb       (MMF waveform, harmonic amplitudes)
      │
      ▼
02_pmsm.ipynb              (torque, back-EMF)
```

**Depends on:** nothing (this is the foundation layer).

**Feeds into:** `01_winding_factors.ipynb`, `01_winding_mmf.ipynb`.

## What You Provide (Inputs)

| Parameter | Symbol | Description | Typical Range |
|-----------|--------|-------------|---------------|
| Stator slots | Q | Total slot count | 6 – 72 |
| Poles | P | Number of poles (even) | 2 – 20 |
| Phases | m | Phase count | 3 |
| Layers | layers | 1 = single-layer, 2 = double-layer | 1 or 2 |
| Coil span | w | Coil span in slots | 1 – Q//p |

## What You Get (Outputs)

| Result | Meaning |
|--------|---------|
| `build_star_of_slots` | Phasor angles for each slot |
| `assign_phases` | Phase/direction assignment per slot |
| `build_coil_matrix` | Integer occupancy matrix (layers × Q) |
| `winding_factor_sos` | |kw(ν)| via phasor sum |
| `get_basic_params` | Derived parameters (p, q, t, …) |
| `check_winding` | Is a coil matrix buildable and balanced? |
| `check_symmetry` | Validate symmetry/balance |
| `get_valid_coil_spans` | All mechanically valid coil spans |
| `is_valid_combination` | Boolean feasibility check |

## Abbreviations and Notation

| Symbol | Definition | Units |
|--------|-----------|-------|
| Q | Number of stator slots | — |
| P | Number of poles | — |
| p | Pole pairs, p = P/2 | — |
| m | Number of phases | — |
| t | Winding periodicity, t = gcd(Q, p) | — |
| q | Slots per pole per phase, q = Q/(2mp) | — |
| τp | Pole pitch in slots, τp = Q/P | slots |
| w | Coil span in slots | slots |
| α_e | Electrical slot pitch angle, α_e = p·(2π/Q) | rad |
| ν | Mechanical spatial harmonic order | — |

## Imports and Setup

In [ ]:
#| default_exp winding.sos

In [ ]:
#| export
from __future__ import annotations

import math
from functools import lru_cache
from fractions import Fraction
from math import gcd

import numpy as np

---

## Private Helpers

These internal functions support the SOS algorithm and are not part of the public API.

In [ ]:
#| export
def _lcm(a: int, b: int) -> int:
    """Least common multiple of two positive integers."""
    return a * b // gcd(a, b)


def _angular_distance(a: float, b: float) -> float:
    """
    Smallest angular distance between two angles (radians), in [0, π].
    """
    d = abs(a - b) % (2 * math.pi)
    return min(d, 2 * math.pi - d)

---

## `get_basic_params`: Winding Parameter Summary

### Theory

All downstream calculations start from a handful of derived parameters.
The winding periodicity *t* = gcd(Q, p) is the key symmetry parameter:
a valid winding repeats itself *t* times around the stator.

The slots-per-pole-per-phase *q* = Q/(2mp) classifies the winding:
- q ≥ 1 (integer): **distributed integer-slot winding** — low harmonic content.
- q < 1 (fractional): **fractional-slot concentrated winding (FSCW)** — compact end-turns, higher harmonic content.

**References:** Pyrhönen et al. (2008), §2.1.

In [ ]:
#| export
def get_basic_params(Q: int, P: int, m: int = 3) -> dict:
    """
    Compute basic winding parameters from slot/pole/phase count.

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles (even)
    m : int   Number of phases (default 3)

    Returns
    -------
    dict with keys:
        p       : int   Pole pairs (P // 2)
        t       : int   Winding periodicity = gcd(Q, p)
        q       : Fraction   Slots per pole per phase = Q / (2mp)
        lcm_QP  : int   lcm(Q, P)
        alpha_e : float Electrical slot pitch (radians) = p · 2π/Q
        Q_phase : int   Slots per phase = Q // m  (integer part)
    """
    p = P // 2
    t = gcd(Q, p)
    q = Fraction(Q, 2 * m * p)
    lcm_QP = _lcm(Q, P)
    alpha_e = p * 2 * math.pi / Q
    Q_phase = Q // m
    return dict(p=p, t=t, q=q, lcm_QP=lcm_QP, alpha_e=alpha_e, Q_phase=Q_phase)

In [ ]:
import math
from fractions import Fraction
import numpy as np
from emachines.winding.sos import (get_basic_params, build_star_of_slots, assign_phases, check_winding,
    build_coil_matrix, winding_factor_sos, check_symmetry, get_valid_coil_spans, is_valid_combination)

# Examples
p = get_basic_params(12, 10)
assert (p['p'], p['t']) == (5, 1)
assert get_basic_params(24, 4)['q'] == Fraction(2, 1)

In [ ]:
from emachines.winding.sos import get_basic_params

params = get_basic_params(12, 10)
print("12s/10p (FSCW):")
for k, v in params.items():
    print(f"  {k:10s} = {v}")

print()
params = get_basic_params(24, 4)
print("24s/4p (integer-slot):")
for k, v in params.items():
    print(f"  {k:10s} = {v}")

---

## `build_star_of_slots`: EMF Phasor Angles

### Theory

The **star of slots** (also called the voltage star or EMF star) maps each stator slot to a phasor angle in the electrical domain.

For slot *i*, the electrical phasor angle is:

$$
\alpha_i = i \cdot \alpha_e \mod 2\pi, \qquad \alpha_e = p \cdot \frac{2\pi}{Q}
$$

where *p* = P/2 is the number of pole pairs and *α_e* is the electrical slot pitch.

Plotting all Q phasors reveals the "star" pattern that names the method. Slots whose phasors point in similar directions naturally carry the same phase current.

**Reference:** Bianchi & Bolognani (2002), IEEE Trans. Ind. Appl., 38(5).

In [ ]:
#| export
def build_star_of_slots(Q: int, P: int) -> np.ndarray:
    """
    Compute EMF phasor angles for all Q slots (electrical radians, mod 2π).

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles

    Returns
    -------
    np.ndarray, shape (Q,), dtype float64
        Phasor angles α_i = i · p · 2π/Q mod 2π
    """
    p = P // 2
    alpha_e = p * 2 * math.pi / Q
    return np.array([(i * alpha_e) % (2 * math.pi) for i in range(Q)],
                    dtype=np.float64)

In [ ]:
# Examples
angles = build_star_of_slots(6, 2)
assert np.allclose(angles, np.arange(6) * np.pi / 3)    # 0°, 60°, … 300°

In [ ]:
from emachines.winding.sos import build_star_of_slots
import numpy as np

angles = build_star_of_slots(12, 10)
print("12s/10p phasor angles (degrees):")
for i, a in enumerate(angles):
    print(f"  Slot {i:2d}: {np.degrees(a):6.1f}°")

---

## `assign_phases`: Phase Assignment via Sector Map

### Theory

The 2π electrical circle is divided into **2m equal sectors** (width = π/m each).
Phase j (0-based) has its go belt at j·2π/m and its return belt π further on, so the sectors interleave go (+) and return (−) belts of different phases. For three phases the six 60° sectors read:

| Sector | 0–60° | 60–120° | 120–180° | 180–240° | 240–300° | 300–360° |
|---|---|---|---|---|---|---|
| Phase | +A | −C | +B | −A | +C | −B |

This places the go belts of A, B and C 120° apart, as a balanced three-phase winding requires. The rule needs an odd phase count; even m raises `ValueError`.

Each slot's phasor (from `build_star_of_slots`) is projected onto this sector map:
- The slot is assigned to the phase whose sector contains the phasor.
- Positive sector → forward conductor (+phase index).
- Negative sector → return conductor (−phase index).

For FSCW, multiple slots in the same phase produce phasors close together, forming tight "clusters" in the star diagram.

**Reference:** Pyrhönen et al. (2008), §2.3; Bianchi & Bolognani (2002).

In [ ]:
#| export
def _assign_slot_to_phase(angle: float, m: int) -> int:
    """
    Return the signed phase code for a slot at electrical angle `angle`.

    The electrical circle [0, 2π) is divided into 2m sectors of width π/m.
    Sector k contains angles [k·π/m, (k+1)·π/m). Phase j (0-based) has its
    go belt at j·2π/m and its return belt π further on, so the sectors
    interleave go and return belts of different phases:

    - k even → go conductor of phase k/2:            +(k/2 + 1)
    - k odd  → return conductor of phase (k-m)/2 mod m: -(((k-m)/2) mod m + 1)

    For m = 3 the six 60° sectors read A, -C, B, -A, C, -B.

    Returns
    -------
    int in {+1…+m, -1…-m}

    Raises
    ------
    ValueError
        If m is even. The interleaved belt rule above needs an odd phase count.
    """
    if m < 1 or m % 2 == 0:
        raise ValueError(f"m must be an odd number of phases, got m={m}")
    sector_width = math.pi / m
    # Small epsilon pushes exact-boundary angles into the right sector
    k = int((angle % (2 * math.pi) + 1e-9) / sector_width) % (2 * m)
    if k % 2 == 0:
        return k // 2 + 1
    return -(((k - m) // 2) % m + 1)


In [ ]:
#| export
def assign_phases(Q: int, P: int, m: int = 3) -> np.ndarray:
    """
    Assign each slot to a phase and current direction using the star-of-slots.

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles
    m : int   Number of phases (default 3)

    Returns
    -------
    np.ndarray, shape (Q,), dtype int
        Signed phase codes: +k = phase k forward, −k = phase k return.
        Phase index k ∈ {1, …, m}.

    Raises
    ------
    ValueError
        If m is even.
    """
    angles = build_star_of_slots(Q, P)
    return np.array([_assign_slot_to_phase(a, m) for a in angles], dtype=int)

In [ ]:
# Examples
assert assign_phases(6, 2, m=3).tolist() == [1, -3, 2, -1, 3, -2]    # A, -C, B, -A, C, -B

In [ ]:
from emachines.winding.sos import assign_phases

codes = assign_phases(12, 10)
print("12s/10p phase assignment:")
names = {1: 'A+', 2: 'B+', 3: 'C+', -1: 'A-', -2: 'B-', -3: 'C-'}
for i, c in enumerate(codes):
    print(f"  Slot {i:2d}: {names[c]}")

---

## `check_winding`: Can It Be Wound, and Is It Balanced?

A coil matrix describes a usable m-phase winding only if two conditions hold:

1. **Buildable.** Every coil has a go side and a return side, so each phase needs as many go (+) as return (−) conductors. Otherwise its conductors cannot be paired into coils. Odd Q, for example, can never be wound single-layer.
2. **Balanced.** At the working harmonic ν = p, the phase EMF phasors $E_k = \sum_i s_i\, e^{j p \theta_i}$ (θᵢ = 2πi/Q) have equal magnitude and step by exactly 360°/m.

Counting conductors per phase is not enough: the old 60°-apart layouts had equal counts but were not balanced.

The fundamental winding factor is $k_{w1} = |E_A| / N_A$, with $N_A$ the conductors in phase A.

In [ ]:
#| export
_PHASE_NAMES = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'


def check_winding(matrix: np.ndarray, P: int, m: int = 3) -> dict:
    """
    Check that a coil matrix can be wound into coils and is balanced.

    Parameters
    ----------
    matrix : np.ndarray, shape (layers, Q)
             Signed phase codes, as returned by ``build_coil_matrix``.
    P      : int   Number of poles
    m      : int   Number of phases (default 3)

    Returns
    -------
    dict with keys:
        buildable    : bool   Each phase has as many go as return conductors
        balanced     : bool   Phase EMFs at ν = p equal in magnitude, 360°/m apart
        kw1          : float  Fundamental winding factor |E_A| / N_A
        go, ret      : list   Go and return conductor counts per phase
        phase_angles : list | None
                              Electrical angle of each phase EMF relative to
                              phase A, in degrees (None if the EMF is zero)
        reason       : str    Why the winding fails, or '' if it passes
    """
    matrix = np.atleast_2d(np.asarray(matrix))
    Q = matrix.shape[1]
    p = P // 2
    go = [int((matrix == k).sum()) for k in range(1, m + 1)]
    ret = [int((matrix == -k).sum()) for k in range(1, m + 1)]
    n = [g + r for g, r in zip(go, ret)]

    e = np.exp(1j * p * 2.0 * np.pi * np.arange(Q) / Q)
    sign = np.sign(matrix)
    E = np.array([np.sum(sign * (np.abs(matrix) == k + 1) * e) for k in range(m)])
    mag = np.abs(E)

    reason = ''
    buildable = go == ret
    if not buildable:
        k = next(i for i in range(m) if go[i] != ret[i])
        reason = (f"phase {_PHASE_NAMES[k]} has {go[k]} go and {ret[k]} return conductors, "
                  f"so it can't be wound into coils")

    if mag.min() < 1e-9:
        balanced, phase_angles = False, None
        reason = reason or "phase EMF at the working harmonic is zero"
    else:
        steps = np.angle(E[1:] / E[:-1]) if m > 1 else np.array([])
        balanced = (len(set(n)) == 1
                    and bool(np.allclose(mag, mag[0], rtol=1e-9, atol=1e-9))
                    and bool(np.allclose(steps, 2.0 * np.pi / m, atol=1e-9)))
        phase_angles = [float(a) % 360.0 for a in np.round(np.degrees(np.angle(E / E[0])), 6)]
        if not balanced:
            reason = reason or "phases are not equal in magnitude and 360°/m apart"

    kw1 = float(mag[0] / n[0]) if n[0] else 0.0
    return dict(buildable=buildable, balanced=balanced, kw1=kw1, go=go, ret=ret,
                phase_angles=phase_angles, reason=reason)


In [ ]:
# Examples
assert check_winding(build_coil_matrix(12, 10, layers=2, w=1), 10)['balanced']
assert check_winding(build_coil_matrix(9, 8, layers=1), 8)['reason'] == (
    "phase A has 2 go and 1 return conductors, so it can't be wound into coils")

In [ ]:
from emachines.winding.sos import check_winding, build_coil_matrix

for Q, P, layers, w in [(12, 10, 2, 1), (24, 4, 1, None), (9, 8, 1, None), (9, 8, 2, 1), (24, 4, 2, 12)]:
    r = check_winding(build_coil_matrix(Q, P, layers=layers, w=w), P)
    label = f"{Q}s/{P}p {'single layer' if layers == 1 else f'double layer, w={w}'}"
    status = "✓" if r['buildable'] and r['balanced'] else "✗"
    print(f"  {status} {label:<28} kw1={r['kw1']:.4f}  phases at {r['phase_angles']}  {r['reason']}")

---

## Winding candidates and the default layout (internal)

For a slot/pole/phase combination the star of slots can produce a single-layer winding and a double-layer winding for each coil span. `_winding_options` builds each one, keeps those that pass `check_winding`, and sorts them by fundamental winding factor kw₁, highest first. Ties go to double layer, then to the shorter span (less end-winding copper).

When `w` is not given, spans from 1 up to the pole pitch Q/P, rounded up, are tried; longer spans only repeat shorter ones mirrored.

`_resolve_layout` is how every function in the winding module fills in a missing `layers` or `w`: it takes the first candidate. Calls that give both `layers` and `w`, or `layers=1`, are used exactly as given. `design_windings` (in the design notebook) returns the full candidate list.

In [ ]:
#| export
def _describe_option(layers: int, w: int | None) -> str:
    return "single layer" if layers == 1 else f"double layer, w={w}"


def _winding_options(Q: int, P: int, m: int = 3, layers: int | None = None,
                     w: int | None = None) -> tuple[list, list]:
    """
    Build every star-of-slots winding allowed by the filters and check it.

    Returns
    -------
    accepted : list of (layers, w, kw1, matrix), best kw1 first
    rejected : list of (layers, w, reason)
    """
    if P < 2 or P % 2:
        raise ValueError(f"P must be an even number of poles >= 2, got P={P}")
    if layers not in (None, 1, 2):
        raise ValueError(f"layers must be 1 or 2, got layers={layers}")
    if layers == 1 and w is not None:
        raise ValueError("w applies to double-layer windings only; "
                         "a single-layer winding has no free coil span")
    if w is not None and not 1 <= w < Q:
        raise ValueError(f"w must be between 1 and Q-1={Q - 1}, got w={w}")

    options: list[tuple[int, int | None]] = []
    if layers in (None, 1) and w is None:
        options.append((1, None))
    if layers in (None, 2):
        spans = [w] if w is not None else range(1, math.ceil(Q / P) + 1)
        options += [(2, s) for s in spans]

    accepted, rejected = [], []
    for lyr, span in options:
        matrix = build_coil_matrix(Q, P, m, layers=lyr, w=span)
        r = check_winding(matrix, P, m)
        if r['buildable'] and r['balanced']:
            accepted.append((lyr, span, r['kw1'], matrix))
        else:
            rejected.append((lyr, span, r['reason']))
    accepted.sort(key=lambda a: (-round(a[2], 12), -a[0], a[1] or 0))
    return accepted, rejected


def _no_winding_error(Q: int, P: int, m: int, layers, w, rejected) -> ValueError:
    why = "; ".join(f"{_describe_option(l, s)}: {r}" for l, s, r in rejected)
    filt = "".join(f", {k}={v}" for k, v in (("layers", layers), ("w", w)) if v is not None)
    return ValueError(f"No buildable, balanced winding for {Q}s/{P}p (m={m}{filt}). {why}")


def _resolve_layout(Q: int, P: int, m: int = 3, layers: int | None = None,
                    w: int | None = None) -> tuple[int, int | None]:
    """
    Fill in a missing ``layers`` or double-layer ``w`` with the best candidate.

    ``layers=1``, or ``layers=2`` with ``w`` given, is returned unchanged.
    """
    if layers == 1 or (layers == 2 and w is not None):
        return layers, w
    accepted, rejected = _winding_options(Q, P, m, layers, w)
    if not accepted:
        raise _no_winding_error(Q, P, m, layers, w, rejected)
    return accepted[0][0], accepted[0][1]


---

## `build_coil_matrix`: Conductor Occupancy Matrix

### Theory

A coil connects a **go** conductor in one slot to a **return** conductor `w` slots away (coil span *w*).
For a single-layer winding, only go conductors are placed; the return side falls in a dedicated slot.
For a double-layer winding, every slot has two conductor layers — a go side from one coil and a return from another.

The coil matrix shape is `(layers, Q)`:
- Value `+k` : phase k, forward current direction.
- Value `−k` : phase k, return current direction.
- Value `0`  : empty slot (only possible in single-layer windings).

This matrix is the single interface consumed by `winding_factors.py` and `mmf.py`.

**References:** Hanselman (2003), Ch. 4; Pyrhönen et al. (2008), §2.3.

In [ ]:
#| export
def build_coil_matrix(
    Q: int,
    P: int,
    m: int = 3,
    layers: int | None = None,
    w: int | None = None,
) -> np.ndarray:
    """
    Build the (layers × Q) conductor occupancy matrix.

    Parameters
    ----------
    Q      : int         Number of stator slots
    P      : int         Number of poles
    m      : int         Number of phases (default 3)
    layers : int | None  1 (single-layer) or 2 (double-layer). None → the best
                         buildable, balanced candidate (see ``design_windings``).
    w      : int | None  Double-layer coil span in slots. None → the best span.
                         Ignored for ``layers=1``.

    Returns
    -------
    np.ndarray, shape (layers, Q), dtype int
        +k = phase k forward, −k = phase k return, 0 = empty.
    """
    layers, w = _resolve_layout(Q, P, m, layers, w)

    phase_codes = assign_phases(Q, P, m)   # go-conductor assignment

    matrix = np.zeros((layers, Q), dtype=int)

    if layers == 1:
        # Place go conductors in all Q slots (single-layer → no return needed here;
        # the return side is implicit in the slot diametrically opposite).
        for i in range(Q):
            matrix[0, i] = phase_codes[i]
    else:
        # Double-layer: top layer = go conductors, bottom layer = return conductors
        # shifted by coil span w.
        assert w is not None  # filled in by _resolve_layout for layers=2
        for i in range(Q):
            go_code  = phase_codes[i]
            ret_slot = (i + w) % Q
            matrix[0, i]        = +go_code
            matrix[1, ret_slot] = -go_code

    return matrix

In [ ]:
# Examples
assert build_coil_matrix(6, 2, m=3, layers=1).tolist() == [[1, -3, 2, -1, 3, -2]]
assert build_coil_matrix(6, 2, m=3, layers=2, w=3).shape == (2, 6)

In [ ]:
from emachines.winding.sos import build_coil_matrix
import numpy as np

print("6s/2p single-layer:")
m1 = build_coil_matrix(6, 2, m=3, layers=1)
print(m1)

print()
print("12s/10p double-layer:")
m2 = build_coil_matrix(12, 10, m=3, layers=2)
print("Top layer:   ", m2[0])
print("Bottom layer:", m2[1])

print()
print("Interpretation: +1=A forward, -1=A return, +2=B forward, -2=B return, etc.")

---

## `winding_factor_sos`: Winding Factor via Phasor Sum

### Theory

The winding factor for mechanical harmonic ν is the magnitude of the normalised phasor sum of all conductors of one phase:

$$
k_{w\nu} = \frac{1}{N_k} \left| \sum_{i,\text{lyr}} s_{i,\text{lyr}} \cdot e^{j\nu \frac{2\pi i}{Q}} \right|
$$

where $s_{i,\text{lyr}} \in \{+1, -1\}$ is the conductor direction, and $N_k$ is the total conductor count per phase. The result is averaged over all m phases.

This phasor method works for **all winding types** — integer-slot, fractional-slot, and FSCW — without needing separate kp and kd formulas.

**References:** Müller, Vogt & Ponick (2008), eq. (3.65); Bianchi & Bolognani (2002).

In [ ]:
#| export
def winding_factor_sos(
    nu: int,
    Q: int,
    P: int,
    m: int = 3,
    layers: int | None = None,
    w: int | None = None,
) -> float:
    """
    Winding factor |kw(ν)| via the star-of-slots phasor method.

    Works for integer-slot, fractional-slot, and FSCW windings.

    Parameters
    ----------
    nu     : int         Mechanical harmonic order
    Q      : int         Number of stator slots
    P      : int         Number of poles
    m      : int         Number of phases (default 3)
    layers : int         1 or 2 (default 1)
    w      : int | None  Coil span. None → Q // (P // 2).

    Returns
    -------
    float   |kw(ν)| ∈ [0, 1]
    """
    layers, w = _resolve_layout(Q, P, m, layers, w)
    matrix = build_coil_matrix(Q, P, m, layers, w)
    slots = np.arange(Q, dtype=np.float64)
    phasors = np.exp(1j * nu * 2.0 * np.pi * slots / Q)

    kw_vals = []
    for k in range(m):
        ph = k + 1
        total = 0j
        n_cond = 0
        for lyr in range(layers):
            fwd = matrix[lyr] == +ph
            ret = matrix[lyr] == -ph
            total += phasors[fwd].sum() - phasors[ret].sum()
            n_cond += int(fwd.sum()) + int(ret.sum())
        if n_cond > 0:
            kw_vals.append(abs(total) / n_cond)

    return float(np.mean(kw_vals)) if kw_vals else 0.0

In [ ]:
# Examples
assert math.isclose(winding_factor_sos(5, 12, 10, layers=2), 0.9330, abs_tol=1e-4)
assert math.isclose(winding_factor_sos(4, 12, 8, layers=2), 0.8660, abs_tol=1e-4)

In [ ]:
from emachines.winding.sos import winding_factor_sos

configs = [
    (12, 10, 2, "12s/10p FSCW  — ν=5 working"),
    (12,  8, 2, "12s/8p  FSCW  — ν=4 working"),
    ( 9,  8, 2, "9s/8p   FSCW  — ν=4 working"),
    (24,  4, 2, "24s/4p  ISW   — ν=2 working"),
]

print(f"{'Configuration':<35}  kw at working ν")
print("-" * 55)
for Q, P, layers, label in configs:
    p = P // 2
    kw = winding_factor_sos(p, Q, P, layers=layers)
    print(f"  {label:<33}  {kw:.4f}")

---

## `check_symmetry`: Validate Winding Balance

### Theory

A valid three-phase winding must be **balanced**: each phase must carry the same number of conductors, and the total slot count must divide evenly.

The check verifies:
1. Q is divisible by m (equal slots per phase).
2. The total period t = gcd(Q, p) > 0.
3. The number of conductors per phase is equal across all m phases.

**Reference:** Pyrhönen et al. (2008), §2.1.

In [ ]:
#| export
def check_symmetry(Q: int, P: int, m: int = 3, layers: int | None = None,
                   w: int | None = None) -> dict:
    """
    Check whether the star-of-slots winding for (Q, P, m) is usable.

    Parameters
    ----------
    Q      : int         Number of stator slots
    P      : int         Number of poles
    m      : int         Number of phases (default 3)
    layers : int | None  1 or 2. None → the best candidate, or single layer
                         if no candidate exists.
    w      : int | None  Double-layer coil span. None → the best span.

    Returns
    -------
    dict with keys:
        balanced     : bool   Phase EMFs equal in magnitude and 360°/m apart
        buildable    : bool   Each phase has as many go as return conductors
        conductors   : list   Conductor count per phase [n_A, n_B, n_C, …]
        periodicity  : int    Winding periodicity t = gcd(Q, p)
        q            : Fraction   Slots per pole per phase
        valid        : bool   balanced and buildable and Q % m == 0
        layers, w    : int, int | None   The layout that was checked
        kw1          : float  Fundamental winding factor
        reason       : str    Why it fails, or ''
    """
    p = P // 2
    t = gcd(Q, p)
    q = Fraction(Q, 2 * m * p)

    if layers is None or (layers == 2 and w is None):
        try:
            layers, w = _resolve_layout(Q, P, m, layers, w)
        except ValueError:
            layers, w = (1, None) if layers is None else (2, max(1, Q // P))

    matrix = build_coil_matrix(Q, P, m, layers, w)
    r = check_winding(matrix, P, m)
    conductors = [g + rr for g, rr in zip(r['go'], r['ret'])]
    valid = r['balanced'] and r['buildable'] and (Q % m == 0)

    return dict(balanced=r['balanced'], buildable=r['buildable'], conductors=conductors,
                periodicity=t, q=q, valid=valid, layers=layers, w=w, kw1=r['kw1'],
                reason=r['reason'])


In [ ]:
# Examples
r = check_symmetry(12, 10, layers=2)
assert (r['balanced'], r['valid']) == (True, True)
r = check_symmetry(9, 8, layers=1)
assert (r['buildable'], r['valid']) == (False, False)

In [ ]:
from emachines.winding.sos import check_symmetry

for Q, P, layers in [(12, 10, 2), (12, 8, 2), (9, 8, 2), (9, 8, 1), (24, 4, 1), (6, 2, 1)]:
    r = check_symmetry(Q, P, layers=layers)
    status = "✓" if r['valid'] else "✗"
    print(f"  {status} {Q}s/{P}p layers={layers}  balanced={r['balanced']}  buildable={r['buildable']}  "
          f"conductors={r['conductors']}  q={r['q']}  t={r['periodicity']}  {r['reason']}")

---

## `get_valid_coil_spans`: Valid Coil Span Enumeration

### Theory

Not all coil spans produce a valid balanced winding. The valid spans are those that maintain winding symmetry — specifically those where the span does not coincide with a phase boundary, causing phase short-circuiting.

For FSCW (q < 1), the only valid span is typically w = 1 (tooth coil).
For integer-slot windings, spans from 1 to `full_pitch` (= Q//p) are candidates; the full-pitch span maximises kw.

The function returns all spans that produce a balanced winding (equal conductors per phase).

In [ ]:
#| export
def get_valid_coil_spans(Q: int, P: int, m: int = 3) -> list[int]:
    """
    Return all coil spans (in slots) that yield a balanced winding.

    Evaluates double-layer spans 1 … max(Q//P, Q//p) and keeps those that
    pass ``check_winding`` (phase EMFs equal and 360°/m apart).

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles
    m : int   Number of phases (default 3)

    Returns
    -------
    list[int]   Valid coil spans in ascending order (empty if none).
    """
    p = P // 2
    valid = []
    for w in range(1, max(Q // P, Q // p, 1) + 1):
        if w >= Q:
            break
        r = check_winding(build_coil_matrix(Q, P, m, layers=2, w=w), P, m)
        if r['buildable'] and r['balanced']:
            valid.append(w)
    return valid


In [ ]:
# Examples
assert get_valid_coil_spans(12, 10) == [1, 2]
assert get_valid_coil_spans(24, 4) == list(range(1, 12))

In [ ]:
from emachines.winding.sos import get_valid_coil_spans

for Q, P in [(12, 10), (12, 8), (9, 8), (24, 4), (36, 6)]:
    spans = get_valid_coil_spans(Q, P)
    print(f"  {Q}s/{P}p  valid spans: {spans}  (pole pitch = {Q / P:.2f} slots)")

---

## `is_valid_combination`: Feasibility Check

Returns `True` if a (Q, P, m) combination can produce a balanced m-phase winding.
This is the entry-point for design sweeps and UI filtering.

In [ ]:
#| export
def is_valid_combination(Q: int, P: int, m: int = 3) -> bool:
    """
    Return True if (Q, P, m) supports a balanced m-phase winding.

    A combination is valid when at least one star-of-slots winding (single
    layer, or double layer with some coil span) can be wound into coils and
    is balanced. P must be even and ≥ 2, and m odd.

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles
    m : int   Number of phases (default 3)

    Returns
    -------
    bool
    """
    if P < 2 or P % 2 != 0:
        return False
    if Q < m:
        return False
    try:
        accepted, _ = _winding_options(Q, P, m)
    except ValueError:
        return False
    return bool(accepted) and Q % m == 0

In [ ]:
# Examples
assert is_valid_combination(12, 10)
assert not is_valid_combination(12, 10, m=5)
assert not is_valid_combination(7, 4)

In [ ]:
from emachines.winding.sos import is_valid_combination

print("Feasibility sweep (Q=6…18, P=2…12, m=3):")
for Q in range(6, 19, 3):
    row = []
    for P in range(2, 13, 2):
        v = is_valid_combination(Q, P)
        row.append(f"P={P}:{'✓' if v else '✗'}")
    print(f"  Q={Q:2d}: {' '.join(row)}")

---

## References

1. Pyrhönen, J., Jokinen, T., & Hrabovcová, V. (2008). *Design of Rotating Electrical Machines*. Wiley. §2.1–2.4.
2. Hanselman, D.C. (2003). *Brushless Permanent Magnet Motor Design*, 2nd ed. Writer's Collective. Ch. 4.
3. Bianchi, N., & Bolognani, S. (2002). Design techniques for reducing cogging torque in surface-mounted PM motors. *IEEE Trans. Ind. Appl.*, 38(5), 1259–1265.
4. Müller, G., Vogt, K., & Ponick, B. (2008). *Berechnung elektrischer Maschinen*. Wiley-VCH. §3.4.
5. SWAT-EM: [https://github.com/bayonet222/swat-em](https://github.com/bayonet222/swat-em)

---

## Tests

In [ ]:
#| hide
import math
import numpy as np
from emachines.winding.sos import (
    get_basic_params, build_star_of_slots, assign_phases,
    build_coil_matrix, winding_factor_sos, check_symmetry,
    get_valid_coil_spans, is_valid_combination, check_winding,
)

# ── get_basic_params ──────────────────────────────────────────────────────────
p = get_basic_params(12, 10)
assert p['p'] == 5 and p['t'] == 1, f"12s/10p params wrong: {p}"
p24 = get_basic_params(24, 4)
assert p24['p'] == 2 and p24['q'] == 2, f"24s/4p params wrong: {p24}"
print("✓ get_basic_params")

# ── build_star_of_slots ───────────────────────────────────────────────────────
angles = build_star_of_slots(6, 2)
assert angles.shape == (6,), "shape"
assert math.isclose(angles[0], 0.0) and math.isclose(angles[3], math.pi), "6s/2p angles"
print("✓ build_star_of_slots")

# ── assign_phases ─────────────────────────────────────────────────────────────
codes = assign_phases(6, 2, m=3)
assert list(codes) == [1, -3, 2, -1, 3, -2], f"6s/2p assignment wrong: {codes}"
codes12 = assign_phases(12, 10)
assert len(codes12) == 12, "length"
assert set(abs(c) for c in codes12) == {1, 2, 3}, "phases 1-3 only"
# 24s/4p, q=2: textbook belts A A -C -C B B -A -A C C -B -B, repeated per pole pair
assert list(assign_phases(24, 4)) == [1, 1, -3, -3, 2, 2, -1, -1, 3, 3, -2, -2] * 2, "24s/4p belts"
# Even phase counts are rejected
for m_even in (2, 4, 6):
    try:
        assign_phases(12, 4, m=m_even)
        raise AssertionError(f"m={m_even} should raise ValueError")
    except ValueError:
        pass
print("✓ assign_phases")

# ── build_coil_matrix ─────────────────────────────────────────────────────────
m1 = build_coil_matrix(6, 2, layers=1)
assert m1.shape == (1, 6), f"shape {m1.shape}"
assert m1.tolist() == [[1, -3, 2, -1, 3, -2]], f"6s/2p SL wrong: {m1}"
m2 = build_coil_matrix(12, 10, layers=2)
assert m2.shape == (2, 12), f"shape {m2.shape}"
# Each layer should have all 12 slots occupied (no zeros for DL FSCW)
assert not (m2 == 0).any(), "DL matrix has empty slots"
print("✓ build_coil_matrix")

# ── winding_factor_sos ────────────────────────────────────────────────────────
kw_12_10 = winding_factor_sos(5, 12, 10, layers=2)
assert math.isclose(kw_12_10, 0.9330, abs_tol=1e-3), f"12s/10p kw(5)={kw_12_10:.4f}"
kw_12_8  = winding_factor_sos(4, 12, 8, layers=2)
assert math.isclose(kw_12_8, 0.8660, abs_tol=1e-3), f"12s/8p kw(4)={kw_12_8:.4f}"
assert 0 < winding_factor_sos(2, 24, 4, layers=1) <= 1.0, "24s/4p kw out of range"
print("✓ winding_factor_sos")

# ── check_symmetry ────────────────────────────────────────────────────────────
r = check_symmetry(12, 10, layers=2)
assert r['balanced'] and r['valid'], f"12s/10p should be valid: {r}"
r_invalid = check_symmetry(7, 4)
assert not r_invalid['valid'], "7s/4p should be invalid"
print("✓ check_symmetry")

# ── get_valid_coil_spans ──────────────────────────────────────────────────────
spans_12_10 = get_valid_coil_spans(12, 10)
assert 1 in spans_12_10, f"12s/10p valid spans should include 1: {spans_12_10}"
spans_24_4 = get_valid_coil_spans(24, 4)
assert 6 in spans_24_4, f"full pitch (6) should be valid for 24s/4p: {spans_24_4}"
assert 12 not in spans_24_4, "w=12 is two pole pitches: phase EMF is zero"
print("✓ get_valid_coil_spans")

# ── is_valid_combination ──────────────────────────────────────────────────────
assert is_valid_combination(12, 10) == True
assert is_valid_combination(7, 4) == False
assert is_valid_combination(12, 10, m=5) == False
print("✓ is_valid_combination")

print()
print("✓ All SOS tests passed")

# ── Parameterised checks across reference combinations ────────────────────────
_ref_combos = [
    # (Q,  P, layers, nu_working, kw_expected, abs_tol, label)
    (12, 10, 2, 5, 0.9330, 1e-3, '12s/10p FSCW'),
    (12, 14, 2, 7, 0.9330, 1e-3, '12s/14p FSCW'),
    (12,  8, 2, 4, 0.8660, 1e-3, '12s/8p  FSCW'),
    (24,  4, 2, 2, 0.9659, 1e-3, '24s/4p  ISW'),
    (36,  8, 2, 4, 0.9452, 1e-3, '36s/8p  ISW'),
]

for Q, P, layers, nu_w, kw_exp, tol, label in _ref_combos:
    # is_valid
    assert is_valid_combination(Q, P), f"{label}: expected valid combination"
    # check_symmetry balanced
    sym = check_symmetry(Q, P, layers=layers)
    assert sym["balanced"], f"{label}: expected balanced winding"
    # build_coil_matrix shape
    mat = build_coil_matrix(Q, P, layers=layers)
    assert mat.shape == (layers, Q), f"{label}: matrix shape {mat.shape}"
    # winding_factor_sos at working harmonic
    kw = winding_factor_sos(nu_w, Q, P, layers=layers)
    assert math.isclose(kw, kw_exp, abs_tol=tol),         f"{label}: kw(nu={nu_w})={kw:.4f}, expected {kw_exp}"
    print(f"pass  {label}: kw(nu={nu_w})={kw:.4f}")


# ── Phase balance: go belts 360°/m apart at the working harmonic ──────────────
# For each phase, sum the conductor phasors at ν = p. In a balanced m-phase
# winding the phase phasors have equal magnitude and step by 2π/m.
def _phase_phasors(Q, P, m, layers, w=None):
    mat = build_coil_matrix(Q, P, m=m, layers=layers, w=w)
    theta = 2 * np.pi * np.arange(Q) / Q
    p = P // 2
    return np.array([
        sum(np.sign(mat[l, i]) * np.exp(-1j * p * theta[i])
            for l in range(layers) for i in range(Q) if abs(mat[l, i]) == k + 1)
        for k in range(m)
    ])

for Q, P, m, layers, w in [(6, 2, 3, 1, None), (24, 4, 3, 1, None), (36, 4, 3, 2, 8),
                           (12, 10, 3, 2, 1), (12, 14, 3, 2, 1), (36, 8, 3, 2, None),
                           (20, 4, 5, 1, None)]:
    ph = _phase_phasors(Q, P, m, layers, w)
    assert np.allclose(abs(ph), abs(ph[0])), f"{Q}s/{P}p m={m}: unequal phase magnitudes"
    steps = np.angle(ph[1:] / ph[:-1])
    # Phases step by -2π/m in space at ν = p, matching currents i_k = cos(θe - 2πk/m)
    assert np.allclose(steps, -2 * np.pi / m), f"{Q}s/{P}p m={m}: phase steps {np.degrees(steps)}°"
    print(f"pass  {Q}s/{P}p m={m}: phases {np.round(np.degrees(np.angle(ph / ph[0])) % 360, 1)}°")


# ── check_winding: buildable and balanced ─────────────────────────────────────
r = check_winding(build_coil_matrix(9, 8, layers=1), 8)
assert not r['buildable'] and r['go'][0] == 2 and r['ret'][0] == 1, r
r = check_winding(build_coil_matrix(9, 8, layers=2, w=1), 8)
assert r['buildable'] and r['balanced'] and math.isclose(r['kw1'], 0.9452, abs_tol=1e-4), r
assert r['phase_angles'] == [0.0, 120.0, 240.0], r['phase_angles']
r = check_winding(build_coil_matrix(24, 4, layers=2, w=12), 4)
assert not r['balanced'] and 'zero' in r['reason'], r
# The old 60°-apart layout had equal conductor counts but is not balanced
old_6s2p = np.array([[1, 2, 3, -1, -2, -3]])
r = check_winding(old_6s2p, 2)
assert r['buildable'] and not r['balanced'], r
assert not check_symmetry(9, 8, layers=1)['valid']
assert is_valid_combination(9, 8), "9s/8p has a double-layer winding"
print("✓ check_winding")

# kw1 from check_winding equals winding_factor_sos at ν = p
for Q, P, layers, w in [(12, 10, 2, 1), (24, 4, 1, None), (36, 8, 2, 4), (15, 8, 2, 2)]:
    r = check_winding(build_coil_matrix(Q, P, layers=layers, w=w), P)
    assert math.isclose(r['kw1'], winding_factor_sos(P // 2, Q, P, layers=layers, w=w), abs_tol=1e-12)
print("✓ check_winding kw1 matches winding_factor_sos")
